In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

### Dimensionality Reduction
In the process of training models we feature engineering find some datasets with a lot of features.most of these features would not contribute signitificaly to predictions,making the training process slow.

In [2]:
from sklearn.datasets import load_wine

data = load_wine()
x = data.data
y = data.target


Obtain 2 principals components of dataset:

In [3]:
x_centered = x - x.mean(axis=0)
u,s,vt = np.linalg.svd(x_centered)
comp1 = vt.T[:,0]
comp2 = vt.T[:,1]

print(comp1,comp2)

[-1.65926472e-03  6.81015556e-04 -1.94905742e-04  4.67130058e-03
 -1.78680075e-02 -9.89829680e-04 -1.56728830e-03  1.23086662e-04
 -6.00607792e-04 -2.32714319e-03 -1.71380037e-04 -7.04931645e-04
 -9.99822937e-01] [-1.20340617e-03 -2.15498184e-03 -4.59369254e-03 -2.64503930e-02
 -9.99344186e-01 -8.77962152e-04  5.18507284e-05  1.35447892e-03
 -5.00440040e-03 -1.51003530e-02  7.62673115e-04  3.49536431e-03
  1.77738095e-02]


contrary: projecting a data in a hyperplane of other components

In [ ]:
#3 components:
w3 = vt.T[:,:3] # 3d hyperplane

x3d = x_centered.dot(w3)
print(x3d) # --> 3d vectors 

In [ ]:
#using sklearn:
from sklearn.decomposition import PCA

pca = PCA(n_components=4)
x4d = pca.fit_transform(x)
print(x4d) # --> 4d project

we can see the influence of variance for each component using explained variance ratio:

In [6]:
pca.explained_variance_ratio_

array([9.98091230e-01, 1.73591562e-03, 9.49589576e-05, 5.02173562e-05])

Using it,you can measure the number of dimensions to preserve to keep a determined variance:

In [7]:
var = np.cumsum(pca.explained_variance_ratio_) #sum ele in axis array
dim = np.argmax(var >= 0.90) + 1

#you could put this limit of variance when selecting the components:

pca2 = PCA(n_components=0.95)
pca3 = PCA(n_components=dim)

Inverse transform helps when you want to reset a projection to its original dimension:

In [9]:
x_90p = pca2.fit_transform(x)
x_100P = pca2.inverse_transform(x_90p)

pca solver: svd-randomizer: less computaion to find n k components

### Incremental Pca:
As Pca uses the whole trainig set,it becomes hard to train new data.

The incremnetal pca (IPCA) has developed for help us in that cases.this divide the data in batchs like stochastic models and is good for online training.

In [ ]:
from sklearn.decomposition import IncrementalPCA

n_batchs = 400
ipca = IncrementalPCA(n_components=3)

for x_batch in np.array_split(x,n_batchs):
    ipca.partial_fit(x) # online training 

#projecting after find patterns:
ipca.fit_transform(x)

As in Suport Vector Machines,we also have Kernel model of Pca.this model allows us compute projections in complex dimensional spaces:

In [11]:
from sklearn.decomposition import KernelPCA

k_pca = KernelPCA(n_components=2,kernel='rbf')


We could save a the original dimension saving a pre-image of data,allowing the reconstruction of a space:

In [ ]:

k_pca = KernelPCA(n_components=2,kernel='rbf',fit_inverse_transform=True)
#-->save the dim of data training 

x_reduced = k_pca.fit_transform(x)
x_original_dim = k_pca.inverse_transform(x_reduced)


and you can evaluate how well the model project:

In [43]:
from sklearn.metrics import mean_squared_error
print(mean_squared_error(x,x_original_dim))

7545.882584490346


In [19]:
print(x)

[[1.423e+01 1.710e+00 2.430e+00 ... 1.040e+00 3.920e+00 1.065e+03]
 [1.320e+01 1.780e+00 2.140e+00 ... 1.050e+00 3.400e+00 1.050e+03]
 [1.316e+01 2.360e+00 2.670e+00 ... 1.030e+00 3.170e+00 1.185e+03]
 ...
 [1.327e+01 4.280e+00 2.260e+00 ... 5.900e-01 1.560e+00 8.350e+02]
 [1.317e+01 2.590e+00 2.370e+00 ... 6.000e-01 1.620e+00 8.400e+02]
 [1.413e+01 4.100e+00 2.740e+00 ... 6.100e-01 1.600e+00 5.600e+02]]


In [20]:
print(x_original_dim)

[[ 12.94317857   2.3234315    2.35654481 ...   0.95359841   2.60068259
  744.59545665]
 [ 12.94277433   2.32282746   2.35656323 ...   0.95364661   2.60070055
  744.79392667]
 [ 12.94317916   2.32343243   2.35654479 ...   0.95359832   2.60068252
  744.59517038]
 ...
 [ 12.94283576   2.32289463   2.35655608 ...   0.95364387   2.60072151
  744.75939912]
 [ 12.94283409   2.32289205   2.35655612 ...   0.95364409   2.60072171
  744.76017442]
 [ 12.94255183   2.32246646   2.35656378 ...   0.95367897   2.60074794
  744.89040973]]


### Locally Linear Embedding
Instead of projections,lle meansure the relation between training set neighborns and search for a low dimension which keep the aproaching variance between data.

In [14]:
from sklearn.manifold import LocallyLinearEmbedding

lle = LocallyLinearEmbedding(n_components=2,n_neighbors=20)
x_lle = lle.fit_transform(x)

Comparing KernelIpca with LLE:

In [17]:
print(mean_squared_error(x_reduced,x_lle))

0.012796097781580614
